In [2]:
# ============================================
# FRAME EXTRACTION FROM VIDEOS (GOOGLE COLAB)
# Reads videos from Google Drive and saves
# extracted frames back to Drive by class.
# ============================================

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import cv2
from pathlib import Path

In [4]:
# -----------------------------
# SET YOUR BASE PROJECT FOLDER
# -----------------------------
BASE_DIR = "/content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization"

RAW_VIDEOS_DIR = os.path.join(BASE_DIR, "raw_videos")
EXTRACTED_FRAMES_DIR = os.path.join(BASE_DIR, "extracted_frames")

os.makedirs(EXTRACTED_FRAMES_DIR, exist_ok=True)

print("RAW_VIDEOS_DIR:", RAW_VIDEOS_DIR)
print("EXTRACTED_FRAMES_DIR:", EXTRACTED_FRAMES_DIR)

RAW_VIDEOS_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/raw_videos
EXTRACTED_FRAMES_DIR: /content/drive/MyDrive/UofT (Google Drive)/MIE1076/AI_Localisation/residence_localization/extracted_frames


In [5]:
# -----------------------------
# CLASS FOLDERS TO PROCESS
# -----------------------------
class_names = [
    "floor1_hallway",
    "floor2_hallway",
    "floor3_hallway",
    "front_lobby",
    "laundry",
    "ccu_lounge",
    "floor1_elevator_landmark",
    "floor2_elevator_landmark",
    "floor3_elevator_landmark"
]

In [6]:
# -----------------------------
# EXTRACTION SETTINGS
# -----------------------------
# Save 1 frame every `seconds_between_frames`
seconds_between_frames = 0.25

# Supported video extensions
video_extensions = {".mp4", ".mov", ".avi", ".mkv"}

In [7]:
def extract_frames_from_video(video_path, output_folder, seconds_between_frames=1.0):
    """
    Extract frames from a single video at a fixed time interval.
    Saves frames into output_folder.

    Returns:
        saved_count (int): Number of frames saved
    """
    os.makedirs(output_folder, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[WARNING] Could not open video: {video_path}")
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps <= 0:
        print(f"[WARNING] Invalid FPS for video: {video_path}")
        cap.release()
        return 0

    frame_interval = max(1, int(round(fps * seconds_between_frames)))

    video_name = Path(video_path).stem
    frame_idx = 0
    saved_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            frame_filename = f"{video_name}_frame_{saved_count:04d}.jpg"
            frame_path = os.path.join(output_folder, frame_filename)
            cv2.imwrite(frame_path, frame)
            saved_count += 1

        frame_idx += 1

    cap.release()
    return saved_count

In [8]:
# -----------------------------
# BATCH EXTRACTION FOR ALL CLASSES
# -----------------------------
summary = {}

for class_name in class_names:
    class_video_dir = os.path.join(RAW_VIDEOS_DIR, class_name)
    class_output_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)

    os.makedirs(class_output_dir, exist_ok=True)

    if not os.path.exists(class_video_dir):
        print(f"[WARNING] Missing class folder: {class_video_dir}")
        summary[class_name] = 0
        continue

    video_files = [
        os.path.join(class_video_dir, f)
        for f in os.listdir(class_video_dir)
        if Path(f).suffix.lower() in video_extensions
    ]

    total_saved = 0

    print(f"\nProcessing class: {class_name}")
    print(f"Found {len(video_files)} video(s)")

    for video_path in sorted(video_files):
        saved = extract_frames_from_video(
            video_path=video_path,
            output_folder=class_output_dir,
            seconds_between_frames=seconds_between_frames
        )
        print(f"  Saved {saved} frame(s) from {os.path.basename(video_path)}")
        total_saved += saved

    summary[class_name] = total_saved

print("\n=== EXTRACTION SUMMARY ===")
for class_name, count in summary.items():
    print(f"{class_name}: {count} frames")


Processing class: floor1_hallway
Found 5 video(s)
  Saved 34 frame(s) from floor1_hallway_1.MOV
  Saved 392 frame(s) from floor1_hallway_2.MOV
  Saved 11 frame(s) from floor1_hallway_3.MOV
  Saved 25 frame(s) from floor1_hallway_4.MOV
  Saved 201 frame(s) from floor1_hallway_5.MOV

Processing class: floor2_hallway
Found 3 video(s)
  Saved 974 frame(s) from floor2_hallway_1.MOV
  Saved 521 frame(s) from floor2_hallway_2.MOV
  Saved 419 frame(s) from floor2_hallway_3.MOV

Processing class: floor3_hallway
Found 4 video(s)
  Saved 49 frame(s) from floor3_hallway_1.MOV
  Saved 264 frame(s) from floor3_hallway_2.MOV
  Saved 68 frame(s) from floor3_hallway_3.MOV
  Saved 632 frame(s) from floor3_hallway_4.MOV

Processing class: front_lobby
Found 3 video(s)
  Saved 64 frame(s) from front_lobby_1.MOV
  Saved 404 frame(s) from front_lobby_2.MOV
  Saved 291 frame(s) from front_lobby_3.MOV

Processing class: laundry
Found 2 video(s)
  Saved 359 frame(s) from laundry_1.MOV
  Saved 26 frame(s) from 

In [9]:
# -----------------------------
# CHECK TOTAL FRAMES SAVED
# -----------------------------
total_frames = 0

for class_name in class_names:
    class_output_dir = os.path.join(EXTRACTED_FRAMES_DIR, class_name)
    if os.path.exists(class_output_dir):
        num_images = len([
            f for f in os.listdir(class_output_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        print(f"{class_name}: {num_images} extracted images")
        total_frames += num_images

print("\nTotal extracted images:", total_frames)

floor1_hallway: 663 extracted images
floor2_hallway: 1914 extracted images
floor3_hallway: 1013 extracted images
front_lobby: 759 extracted images
laundry: 385 extracted images
ccu_lounge: 553 extracted images
floor1_elevator_landmark: 502 extracted images
floor2_elevator_landmark: 526 extracted images
floor3_elevator_landmark: 408 extracted images

Total extracted images: 6723
